In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

train_df = pd.read_csv("../datasets/filled.csv")
test_df = pd.read_csv("../datasets/test.csv")   # separate names, so the training data is never overwritten
print(train_df.shape, test_df.shape)

(8693, 14) (4277, 13)


In [2]:
features = ["CryoSleep", "Age", "VIP", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def prepare(df):
    X = df[features].copy()
    # In test.csv these columns contain gaps, so pandas reads them as text/objects.
    # Turn them into 1/0 (gaps stay NaN and get filled by the imputer below).
    for col in ["CryoSleep", "VIP"]:
        X[col] = X[col].map({True: 1, False: 0, "True": 1, "False": 0})
    return X

X = prepare(train_df)
Y = train_df["Transported"]
X.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0


In [3]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=16)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=16),
)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)

print("Validation accuracy:", round(metrics.accuracy_score(Y_test, Y_pred), 3))
print(metrics.confusion_matrix(Y_test, Y_pred))

Validation accuracy: 0.766
[[814 278]
 [230 852]]


In [5]:
model.fit(X, Y)
test_pred = model.predict(prepare(test_df))

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": test_pred.astype(bool),
})
submission.to_csv("../datasets/submission2.csv", index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
